<a href="https://colab.research.google.com/github/metarun/Mechanistic-Interpretability-Exploration/blob/main/Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Understanding Simple Self-Attention (Without Trainable Weights)**

In this notebook, I'll implement a simplified version of the Self-Attention mechanism. For educational purposes, I'll assume the **Query (Q)**, **Key (K)**, and **Value (V)** matrices are derived directly from the input embeddings without initial linear transformations.

### The Mathematical Foundation
The core formula for scaled dot-product attention is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In a full Transformer layer, these are calculated as:

$$\text{head}_i = \text{Attention}\left(QW_i^Q,\; KW_i^K,\; VW_i^V\right)$$

Example
* d_model = 6   (embedding size, your design choice)
* d_in    = 6   (input to W_Q, same as d_model)
* d_out   = 6   (output of W_Q, same as d_head = d_model/n_heads = 6/1)

### **The Self-Attention Mechanism: Step-by-Step**

The self-attention formula is defined as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

#### **Key Components:**

1.  **Query (Q), Key (K), and Value (V)**:
    *   **Query**: What we are looking for (the 'search term').
    *   **Key**: What each token offers (the 'index' or 'label').
    *   **Value**: The actual information associated with the token.

2.  **Attention Scores ($QK^T$)**: Measures similarity between the Query and all Keys.

3.  **Scaling ($\sqrt{d_k}$)**: Prevents gradients from vanishing by keeping dot-product values manageable.

4.  **Softmax**: Normalizes scores into weights that sum to 1, creating a probability distribution over the sequence.

5.  **Weighted Sum**: Multiplies the weights by the Value vectors to create a context-aware representation.

# **Step by Step Calcluation of Attention Just for 1 Token for now**

## **Data Setup: Defining Our Input Sentance / Tokens**

In [41]:
import pandas as pd

# Define the labels for the input tokens based on the comment in the 'inputs' tensor --> Below
token_labels = [
    "Your",      # x^1
    "journey",   # x^2
    "starts",    # x^3
    "with",      # x^4
    "one",       # x^5
    "step"       # x^6
]
token_labels

['Your', 'journey', 'starts', 'with', 'one', 'step']

## **Data Setup: Initlislising Our Input Tokens with random weights**


In [42]:
import torch
torch.manual_seed(42)
# Define input embeddings for our sample sentence:
# "Your journey starts with one step"
# Each row represents a token, each column represents a dimension (d_model = 3)
inputs = torch.tensor(
     [[0.43, 0.15, 0.89], # 'Your'    (x^1)
      [0.55, 0.87, 0.66], # 'journey' (x^2)
      [0.57, 0.85, 0.64], # 'starts'  (x^3)
      [0.22, 0.58, 0.33], # 'with'    (x^4)
      [0.77, 0.25, 0.10], # 'one'     (x^5)
      [0.05, 0.80, 0.55], # 'step'    (x^6)
    ]
)

print(f"Input shape: {inputs.shape} (6 tokens, 3 dimensions)")

Input shape: torch.Size([6, 3]) (6 tokens, 3 dimensions)


## **Data Setup: Creating Q K V metrices for our calculations for Attention**


In [43]:
K = inputs
Q = inputs
V = inputs

## **Step 1: Calculating the Dot Product Similarity ($QK^T$)**


I first compute the similarity between tokens. I'll start by looking at a single query (the token 'journey') against all other tokens (keys). This dot product serves as a measure of alignment: the higher the score, the more relevant the key is to the query.

In [44]:
# I select the 2nd token ('journey') as my Query (Q)
query_journey = inputs[1]

# For this simplified version, I'll use the transpose of the input matrix as my Keys (K)
# This allows me to perform a dot product between the query and every token in the sequence simultaneously
Kt = K.T

# Calculate attention scores for the 'journey' token
# Score(q, k) = q · k
attention_score_journey = query_journey @ Kt

print("Attention scores for 'journey':", attention_score_journey)

Attention scores for 'journey': tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## **Step 2: Scaling and Normalization**

For this simple example, I assume $d_k=1$ (no scaling). I use the Softmax function to turn the raw scores into weights. This step is crucial because it ensures all weights are positive and sum to 1, effectively creating a probability distribution that dictates how much 'attention' I pay to each token.

In [45]:
# Step 2: Apply Softmax to obtain Attention Weights
# These weights determine how much focus the model puts on other parts of the sentence.

def softmax_naive(x):
    """A basic implementation of the softmax function."""
    return torch.exp(x) / torch.exp(x).sum(dim=0)

# Comparing manual vs. PyTorch implementation

# Applying the manual softmax function to our specific query scores
manual_weights_journey = softmax_naive(attention_score_journey)
print(f"manual_weights_journey -- > {manual_weights_journey}")


# Using PyTorch's optimized softmax for the 'journey' token weights
pytorch_weights_journey = torch.softmax(attention_score_journey, dim=0)
print(f"pytorch_weights_journey -- > {pytorch_weights_journey}")

manual_weights_journey -- > tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
pytorch_weights_journey -- > tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


## **Step 3: Compute the Context Vector**
I multiply the attention weights by the input embeddings to get a single vector

That represents 'journey' in the context of the entire sentence.

**Find context vector / Atention with respect to 2nd input token - journey $$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [46]:
context_vector_journey = pytorch_weights_journey @ V

print("Original 'journey' vector:  ", inputs[1])
print("Context-aware representation:", context_vector_journey)

Original 'journey' vector:   tensor([0.5500, 0.8700, 0.6600])
Context-aware representation: tensor([0.4419, 0.6515, 0.5683])


So our token - Journey was **[0.55, 0.87, 0.66]** but now we have added attention weight and also gave it the context for entire sentance hence now it is transformed and is now more alingend with context of entire sentance hence now it is **[0.4419, 0.6515, 0.5683]**

# **Scaling to the Entire Sequence**

Instead of calculating one token at a time, I can compute the context vectors for the entire sentence simultaneously using matrix multiplication. This is where the efficiency of Transformers truly shines, as I can process the whole sequence in parallel.

* input_query == > inputs
* Key == > inputs ( For this example)
* Value = > inputs ( For this example)

**Lets calculate ($K^T$)**

In [47]:
Kt = K.T
Kt

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

### **Lets calculate ($QK^T$)**

In [48]:
QKt = Q @ Kt
QKt

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [49]:
# Show Attention Score for all inputs with respect to other tokens
QKt_df = pd.DataFrame(QKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(QKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.9995,0.9544,0.9422,0.4753,0.4576,0.6310
journey,0.9544,1.4950,1.4754,0.8434,0.7070,1.0865
starts,0.9422,1.4754,1.4570,0.8296,0.7154,1.0605
with,0.4753,0.8434,0.8296,0.4937,0.3474,0.6565
one,0.4576,0.7070,0.7154,0.3474,0.6654,0.2935
step,0.6310,1.0865,1.0605,0.6565,0.2935,0.9450


## **Lets take Softmax of ($QK^T$)**

In [50]:
SoftMaxQKt = torch.softmax(QKt, dim=1)
SoftMaxQKt

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [51]:
# Show Attention Score for all inputs with respect to other tokens
SoftMaxQKt_df = pd.DataFrame(SoftMaxQKt.numpy(), index=token_labels, columns=token_labels)

print("Attention Score (rows are Queries, columns are dimensions of the context vector):")
display(SoftMaxQKt_df)

Attention Score (rows are Queries, columns are dimensions of the context vector):


,Your,journey,starts,with,one,step
Your,0.209835,0.200581,0.198149,0.124228,0.122049,0.145158
journey,0.138548,0.237891,0.233274,0.123992,0.108182,0.158114
starts,0.139008,0.236921,0.232602,0.124204,0.110800,0.156464
with,0.143527,0.207394,0.204552,0.146192,0.126295,0.172039
one,0.152611,0.195839,0.197491,0.136687,0.187859,0.129514
step,0.138471,0.218364,0.212759,0.142048,0.098806,0.189552


## **Lets find the attention/context vectors for all tokens**

**$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$**

In [52]:
# Multiple SoftMaxQKt with V ==> inputs
AttentionWeights = SoftMaxQKt @ V
AttentionWeights

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [53]:
# Create the DataFrame with correct dimensions
attention_df = pd.DataFrame(AttentionWeights.numpy(), index=token_labels)

print("Attention Weights (rows are Queries, columns are dimensions of the context vector):")
display(attention_df)

Attention Weights (rows are Queries, columns are dimensions of the context vector):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535


This DataFrame `attention_df` now shows the new, context-aware representations for each of your original tokens. For instance, the first row, labeled 'Your', now has a vector `[0.4421, 0.5931, 0.5790]`, which is its updated representation incorporating information from all other tokens in the sentence through the self-attention mechanism.

# **Single step compute self attention weights for entire sentance**

In [54]:
# Compute Attention in a single efficient operation
# 1. inputs @ inputs.T -> Compute all dot product scores (Q * K^T)
# 2. torch.softmax(..., dim=1) -> Normalize scores across rows
# 3. @ inputs -> Multiply weights by values (V)

attention_outputs = torch.softmax(Q @ K.T, dim=1) @ V

print("Full Attention Output Matrix:")
display(attention_outputs)

Full Attention Output Matrix:


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [55]:
# Final Step: Visualizing the results in a readable format
# I create a DataFrame to map the context-aware vectors back to their original tokens.
# This final representation is what a Transformer block would pass to the next layer.
final_results_df = pd.DataFrame(attention_outputs.numpy(), index=token_labels)

print("Context-Aware Token Representations (Output of Self-Attention):")
display(final_results_df)

Context-Aware Token Representations (Output of Self-Attention):


,0,1,2
Your,0.442059,0.593099,0.578989
journey,0.441866,0.651482,0.568309
starts,0.443128,0.649595,0.567073
with,0.430390,0.629828,0.551027
one,0.467102,0.590993,0.526597
step,0.417724,0.650323,0.564535


## **Summary**
By applying self-attention, I have transformed static embeddings into dynamic, context-aware representations. Each token's new vector now contains rich information about its relationship with every other token in the sequence, allowing the model to understand the nuances of the language.

# **Trainable Weights for K Q V For Single Token Step by Step**
* **Single Attention head**
* **Embedding size for our example is Vocab * 2**

In [56]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [57]:
query_journey = inputs[1]
d_in = d_model = 3  # since we have decided above to have 3 features for our tokens
n_heads = 1         # we want single attention head for now
d_out = d_head = d_model / n_heads
d_out = d_head = int(d_out)
print(f" d_in {d_in} \n n_heads {n_heads} \n d_out {d_out} \n d_head {d_head}")


 d_in 3 
 n_heads 1 
 d_out 3 
 d_head 3


In [58]:
torch.manual_seed(42)
W_q = torch.nn.Parameter(torch.randn(d_in, d_out)) # Query weight matrix
W_k = torch.nn.Parameter(torch.randn(d_in, d_out)) # Key weight matrix
W_v = torch.nn.Parameter(torch.randn(d_in, d_out)) # Value weight matrix
W_q, W_k, W_v

(Parameter containing:
 tensor([[ 0.3367,  0.1288,  0.2345],
         [ 0.2303, -1.1229, -0.1863],
         [ 2.2082, -0.6380,  0.4617]], requires_grad=True),
 Parameter containing:
 tensor([[ 0.2674,  0.5349,  0.8094],
         [ 1.1103, -1.6898, -0.9890],
         [ 0.9580,  1.3221,  0.8172]], requires_grad=True),
 Parameter containing:
 tensor([[-0.7658, -0.7506,  1.3525],
         [ 0.6863, -0.3278,  0.7950],
         [ 0.2815,  0.0562,  0.5227]], requires_grad=True))

In [59]:
Q = inputs @ W_q
K = inputs @ W_k
V = inputs @ W_v
Q, K, V

(tensor([[ 2.1446, -0.6809,  0.4837],
         [ 1.8430, -1.3271,  0.2715],
         [ 1.8009, -1.2893,  0.2707],
         [ 0.9364, -0.8335,  0.0959],
         [ 0.5377, -0.2453,  0.1801],
         [ 1.4156, -1.2427,  0.1166]], grad_fn=<MmBackward0>),
 tensor([[ 1.1341,  1.1532,  0.9270],
         [ 1.7453, -0.3033,  0.1241],
         [ 1.7092, -0.2853,  0.1437],
         [ 1.0189, -0.4261, -0.1259],
         [ 0.5792,  0.1216,  0.4577],
         [ 1.4285, -0.5979, -0.3012]], grad_fn=<MmBackward0>),
 tensor([[ 0.0242, -0.3219,  1.1661],
         [ 0.3617, -0.6609,  1.7805],
         [ 0.3270, -0.6705,  1.7812],
         [ 0.3225, -0.3367,  0.9311],
         [-0.3900, -0.6543,  1.2925],
         [ 0.6656, -0.2688,  0.9911]], grad_fn=<MmBackward0>))

## **Step 1: Calculating the Dot Product Similarity ($QK^T$)**

In [60]:
score = Q @ ( K.T )
score

tensor([[2.0954, 4.0095, 3.9294, 2.4144, 1.3808, 3.3249],
        [0.8114, 3.6527, 3.5677, 2.4092, 1.0304, 3.3444],
        [0.8065, 3.5678, 3.4850, 2.3503, 1.0102, 3.2620],
        [0.1896, 1.8989, 1.8520, 1.2972, 0.4849, 1.8071],
        [0.4938, 1.0351, 1.0149, 0.6297, 0.3640, 0.8605],
        [0.2803, 2.8620, 2.7909, 1.9572, 0.7222, 2.7301]],
       grad_fn=<MmBackward0>)

## **Step 2: Scaling**
scores = scores / sqrt(d_head)


In [61]:
import torch # ensure torch is imported
sqrt_d_head = torch.sqrt(torch.tensor(d_out, dtype=torch.float32))
sqrt_d_head
score = score / sqrt_d_head
score

tensor([[1.2098, 2.3149, 2.2687, 1.3940, 0.7972, 1.9197],
        [0.4684, 2.1089, 2.0598, 1.3909, 0.5949, 1.9309],
        [0.4656, 2.0599, 2.0120, 1.3570, 0.5833, 1.8833],
        [0.1095, 1.0963, 1.0693, 0.7489, 0.2799, 1.0433],
        [0.2851, 0.5976, 0.5859, 0.3636, 0.2102, 0.4968],
        [0.1618, 1.6524, 1.6113, 1.1300, 0.4169, 1.5762]],
       grad_fn=<DivBackward0>)

## **Step 3: Sclae with  the Context Vector Softmax of score**

In [62]:
score = torch.softmax(score, dim=-1)
score

tensor([[0.0926, 0.2796, 0.2669, 0.1113, 0.0613, 0.1883],
        [0.0525, 0.2710, 0.2580, 0.1322, 0.0596, 0.2268],
        [0.0546, 0.2690, 0.2564, 0.1332, 0.0614, 0.2254],
        [0.0839, 0.2251, 0.2191, 0.1590, 0.0995, 0.2135],
        [0.1436, 0.1963, 0.1940, 0.1553, 0.1332, 0.1775],
        [0.0564, 0.2503, 0.2402, 0.1484, 0.0728, 0.2319]],
       grad_fn=<SoftmaxBackward0>)

In [63]:
score[:1].sum(dim=1)

tensor([1.0000], grad_fn=<SumBackward1>)

## **Step 3: Compute the Context Vector**

In [64]:
context_vector = score @ V
context_vector

tensor([[ 0.3280, -0.5218,  1.4507],
        [ 0.3539, -0.5134,  1.4281],
        [ 0.3515, -0.5129,  1.4261],
        [ 0.3096, -0.4987,  1.3770],
        [ 0.2542, -0.4933,  1.3553],
        [ 0.3443, -0.5046,  1.4014]], grad_fn=<MmBackward0>)

So our token - Journey was **[0.55, 0.87, 0.66]** but now we have added attention weight and also gave it the context for entire sentance hence now it is transformed and is now more alingend with context of entire sentance hence now it is **[0.8303, 0.0858, 0.0840]**

# **Entire Single Head Attention with 1 step**

In [65]:
query_journey = inputs[1]
d_in = d_model = 3  # since we have decided above to have 3 features for our tokens
n_heads = 1         # we want single attention head for now
d_out = d_head = int(d_model / n_heads)
print(f" d_in {d_in} \n n_heads {n_heads} \n d_out {d_out} \n d_head {d_head}")

 d_in 3 
 n_heads 1 
 d_out 3 
 d_head 3


In [66]:
inputs
# We already set the seed in the previous cell (cZggBgKtxBSO) for W_q, W_k, W_v
# So we will use the same W_q, W_k, W_v parameters and avoid re-initialization.
# If you uncomment the manual_seed and re-initialize here, the values will change.
# torch.manual_seed(42)
# W_q = torch.nn.Parameter(torch.randn(d_model, d_head))
# W_k = torch.nn.Parameter(torch.randn(d_model, d_head))
# W_v = torch.nn.Parameter(torch.randn(d_model, d_head))
# W_q,W_k,W_v

# Compute Q, K, V matrices using the learned weight matrices
Q = inputs @ W_q
K = inputs @ W_k
V = inputs @ W_v

# Calculate the full attention output in one step
# 1. Compute attention scores (Q @ K.T)
# 2. Scale the scores by sqrt(d_out) for stability
# 3. Apply softmax to get attention weights
# 4. Multiply weights by V to get the context vector
context_vector_ =  (torch.softmax((Q @ K.T) / torch.sqrt(torch.tensor(d_out, dtype=torch.float32)), dim= -1)) @ V
context_vector_

tensor([[ 0.3280, -0.5218,  1.4507],
        [ 0.3539, -0.5134,  1.4281],
        [ 0.3515, -0.5129,  1.4261],
        [ 0.3096, -0.4987,  1.3770],
        [ 0.2542, -0.4933,  1.3553],
        [ 0.3443, -0.5046,  1.4014]], grad_fn=<MmBackward0>)

# **Entire Single Head Attention with 1 step + Masking**

Why we need masking?

Because in our example we have 6 tokens **Your journey starts with one step** and **Your** Should never see **Your** see any of the **journey starts with one step**  same with **journey** should never see **starts with one step**

So we need something like below

<img src="https://raw.githubusercontent.com/metarun/images/main/masking.png" width="400" style="vertical-align:middle;">



```
import torch
import pandas as pd

tokens = ["Your", "journey", "starts", "with", "one", "step"]
n = len(tokens)

mask = torch.triu(torch.ones(n, n), diagonal=1).bool()

rows = []
for i in range(n):
    row = []
    for j in range(n):
        row.append("MASKED" if mask[i, j].item() else "SEE")
    rows.append(row)

df = pd.DataFrame(rows, index=tokens, columns=tokens)
print(df)

```


In [67]:
mask = torch.triu(torch.ones(6, 6), diagonal=1).bool()

In [68]:
torch.manual_seed(42)
W_q = torch.nn.Parameter(torch.randn(d_in, d_out))
W_k = torch.nn.Parameter(torch.randn(d_in, d_out))
W_v = torch.nn.Parameter(torch.randn(d_in, d_out))

Q = inputs @ W_q
K = inputs @ W_k
V = inputs @ W_v

scores = Q @ K.T
scores = scores / torch.sqrt(torch.tensor(d_out, dtype=torch.float32))
scores = scores.masked_fill(mask, float('-inf'))
weights = torch.softmax(scores, dim=-1)
context_vector_ = weights @ V
context_vector_

tensor([[ 0.0242, -0.3219,  1.1661],
        [ 0.3069, -0.6059,  1.6807],
        [ 0.3146, -0.6332,  1.7230],
        [ 0.3003, -0.5475,  1.5091],
        [ 0.1654, -0.5417,  1.4339],
        [ 0.3443, -0.5046,  1.4014]], grad_fn=<MmBackward0>)

In [69]:
print(weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1624, 0.8376, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0942, 0.4637, 0.4421, 0.0000, 0.0000, 0.0000],
        [0.1221, 0.3276, 0.3188, 0.2314, 0.0000, 0.0000],
        [0.1746, 0.2387, 0.2359, 0.1889, 0.1620, 0.0000],
        [0.0564, 0.2503, 0.2402, 0.1484, 0.0728, 0.2319]],
       grad_fn=<SoftmaxBackward0>)


# **Multi Head Attention with multipe step + Masking**


In [70]:
torch.manual_seed(42)

d_model = d_in = 8                         # embedding size for each token
n_heads = 2                         # number of attention heads
d_head = d_out = int(d_model/n_heads)      # dimension of each head (d_model / n_heads = 8 / 2 = 4)
n_tokens = 6                        # number of tokens in our input sequence (e.g., "Your journey starts with one step")

In [71]:
# Input — random embeddings for this example
inputs = torch.randn(n_tokens, d_model)
torch.manual_seed(42)

# Output weight matrix (to combine results from multiple heads)
W_o = torch.nn.Parameter(torch.randn(d_model, d_model))

# Head 1 weights for Q, K, V
W_q1 = torch.nn.Parameter(torch.randn(d_model, d_head))
W_k1 = torch.nn.Parameter(torch.randn(d_model, d_head))
W_v1 = torch.nn.Parameter(torch.randn(d_model, d_head))

# Compute Q, K, V for Head 1
Q1 = inputs @ W_q1
K1 = inputs @ W_k1
V1 = inputs @ W_v1

# Calculate attention scores for Head 1
scores = Q1 @ K1.T
scores = scores / torch.sqrt(torch.tensor(d_out, dtype=torch.float32)) # Scale scores
scores = scores.masked_fill(mask, float('-inf')) # Apply masking
weights = torch.softmax(scores, dim=-1) # Get attention weights
context_vector_1 = weights @ V1 # Compute context vector for Head 1

# Head 2 weights for Q, K, V
W_q2 = torch.nn.Parameter(torch.randn(d_model, d_head))
W_k2 = torch.nn.Parameter(torch.randn(d_model, d_head))
W_v2 = torch.nn.Parameter(torch.randn(d_model, d_head))

# Compute Q, K, V for Head 2
Q2 = inputs @ W_q2
K2 = inputs @ W_k2
V2 = inputs @ W_v2

# Calculate attention scores for Head 2
scores = Q2 @ K2.T
scores = scores / torch.sqrt(torch.tensor(d_head, dtype=torch.float32)) # Scale scores
scores = scores.masked_fill(mask, float('-inf')) # Apply masking
weights = torch.softmax(scores, dim=-1) # Get attention weights
context_vector_2 = weights @ V2 # Compute context vector for Head 2

# Concatenate the context vectors from all heads
combined_context_vector_ = torch.cat([context_vector_1,context_vector_2], dim=1)

# Project the combined context vector back to d_model dimension
output = combined_context_vector_ @ W_o

# Add residual connection (inputs + output of attention block)
residual = inputs + output
residual

tensor([[ 20.5555,  20.9446,   3.3351, -33.6472,   1.3231, -15.0576,   1.3838,
         -12.7886],
        [ 17.8764,  21.1060,   2.0420, -32.9452,  -0.0832, -14.3825,   0.6580,
         -10.4215],
        [-12.1792,  -2.0844,  -0.1230,   1.6347,   0.2809,   4.4267,  -6.4354,
          -0.3984],
        [ -9.4709,  -3.5110,  -1.2281,   6.1397,  -1.6039,   8.0218,  -4.7165,
           4.3083],
        [ -2.6957,  -1.1480,  -1.9566,  10.9447,  -4.5050,  10.8658,  -2.5017,
          10.2168],
        [  8.6905,  17.6795,   0.7570, -25.6335,  -1.1695,  -9.6809,  -2.2221,
         -10.5684]], grad_fn=<AddBackward0>)

In [72]:
print(f"inputs shape:           {inputs.shape}")
print(f"combined_context shape: {combined_context_vector_.shape}")
print(f"output shape:           {output.shape}")
print(f"residual shape:         {residual.shape}")

inputs shape:           torch.Size([6, 8])
combined_context shape: torch.Size([6, 8])
output shape:           torch.Size([6, 8])
residual shape:         torch.Size([6, 8])


# **Multi Head Attention with matrix multiplation and reshaping + Masking**


In [73]:
torch.manual_seed(42)

d_model  = 8                         # embedding size for each token
n_heads = 2                          # number of attention heads
d_head = int(d_model/n_heads)        # dimension of each head (d_model / n_heads = 8 / 2 = 4)
n_tokens = 6                         # number of tokens in our input sequence

In [74]:
# Input — random embeddings for this example
torch.manual_seed(42)
inputs = torch.randn(n_tokens, d_model)

# Output weight matrix (to combine results from multiple heads)
W_o = torch.nn.Parameter(torch.randn(d_model, d_model))

# Weight matrices for Q, K, V for ALL heads combined
# These matrices transform the input into Q, K, V for all heads simultaneously
W_q = torch.nn.Parameter(torch.randn(d_model, n_heads * d_head))
W_k = torch.nn.Parameter(torch.randn(d_model, n_heads * d_head))
W_v = torch.nn.Parameter(torch.randn(d_model, n_heads * d_head))

# Compute Q, K, V for all heads
# Q, K, V are initially (n_tokens, n_heads * d_head)
Q = inputs @ W_q
K = inputs @ W_k
V = inputs @ W_v

# Reshape Q, K, V to separate the heads and prepare for batch matrix multiplication
# From (n_tokens, n_heads * d_head) to (n_tokens, n_heads, d_head)
Q = Q.reshape(n_tokens, n_heads, d_head)
K = K.reshape(n_tokens, n_heads, d_head)
V = V.reshape(n_tokens, n_heads, d_head)

# Permute Q, K, V to have heads as the first dimension for batch processing
# From (n_tokens, n_heads, d_head) to (n_heads, n_tokens, d_head)
Q = Q.permute(1, 0, 2)
K = K.permute(1, 0, 2)
V = V.permute(1, 0, 2)

# Transpose K for matrix multiplication (QK^T)
KT  = K.transpose(-1, -2)

# Compute attention scores for all heads simultaneously
# Result is (n_heads, n_tokens, n_tokens)
scores = Q @ KT

# Scale scores
d_head_sqrt = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
scores = scores / d_head_sqrt

# Apply causal masking (prevent tokens from attending to future tokens)
# Mask is broadcasted across heads
scores = scores.masked_fill(mask, float('-inf'))

# Apply softmax to get attention weights
# Result is (n_heads, n_tokens, n_tokens)
weights = torch.softmax(scores, dim=-1)

# Compute the context vector for all heads
# Result is (n_heads, n_tokens, d_head)
context_vector = weights @ V

# Permute and reshape the context vector back to (n_tokens, d_model)
# First, permute from (n_heads, n_tokens, d_head) to (n_tokens, n_heads, d_head)
context_vector = context_vector.permute(1, 0, 2)
# Then, reshape to concatenate the head outputs: (n_tokens, n_heads * d_head) which is (n_tokens, d_model)
context_vector = context_vector.reshape(n_tokens, n_heads * d_head)

# Project the combined context vector back to d_model dimension using W_o
# Result is (n_tokens, d_model)
output = context_vector @ W_o

# Add residual connection (inputs + output of attention block)
residual = inputs + output
residual

tensor([[-10.5909,   2.3432,  -2.1207,  -6.6168,   0.7252,  -2.3350,  11.6954,
         -12.4908],
        [-14.3620,  -2.2513, -13.6243,  -8.7154,  -0.9354,  -3.6114,  21.6141,
          -8.4644],
        [  3.2705,   3.2568,   7.5265,   8.7610,  -2.2385,   8.1775, -14.8287,
           3.1239],
        [  3.3433,   4.6284,   8.3512,   9.5822,  -1.7428,   7.1500, -15.9255,
           2.3841],
        [ -7.2523,   7.5504,  16.6206,   6.1644,   3.7888,   1.8357,  -9.4678,
          -1.9809],
        [-10.4061,   8.3981,  18.3978,   7.1894,   1.0644,   9.9126, -17.4105,
          -1.5298]], grad_fn=<AddBackward0>)

In [75]:
inputs.shape, residual.shape, inputs, residual

(torch.Size([6, 8]),
 torch.Size([6, 8]),
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047],
         [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
         [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,  1.6806],
         [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599],
         [-1.3847, -0.8712, -0.2234,  1.7174,  0.3189, -0.4245,  0.3057, -0.7746],
         [-1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347, -0.4879]]),
 tensor([[-10.5909,   2.3432,  -2.1207,  -6.6168,   0.7252,  -2.3350,  11.6954,
          -12.4908],
         [-14.3620,  -2.2513, -13.6243,  -8.7154,  -0.9354,  -3.6114,  21.6141,
           -8.4644],
         [  3.2705,   3.2568,   7.5265,   8.7610,  -2.2385,   8.1775, -14.8287,
            3.1239],
         [  3.3433,   4.6284,   8.3512,   9.5822,  -1.7428,   7.1500, -15.9255,
            2.3841],
         [ -7.2523,   7.5504,  16.6206,   6.1644,   3.

In [76]:
print(f"Q shape:              {Q.shape}")        # (n_heads, n_tokens, d_head)
print(f"scores shape:         {scores.shape}")   # (n_heads, n_tokens, n_tokens)
print(f"weights shape:        {weights.shape}")  # (n_heads, n_tokens, n_tokens)
print(f"context_vector shape: {context_vector.shape}")  # (n_tokens, d_model)
print(f"residual shape:       {residual.shape}") # (n_tokens, d_model)
print(f"mask shape: {mask.shape}")


Q shape:              torch.Size([2, 6, 4])
scores shape:         torch.Size([2, 6, 6])
weights shape:        torch.Size([2, 6, 6])
context_vector shape: torch.Size([6, 8])
residual shape:       torch.Size([6, 8])
mask shape: torch.Size([6, 6])


## **Multi-Head Attention: Class-based Implementation**

For a more modular and reusable approach, especially when building larger models, it's common to encapsulate the Multi-Head Attention logic within a `torch.nn.Module` class. This allows the attention mechanism to be treated as a distinct layer that can be easily instantiated and integrated.

In [77]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal_mask=True):

        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads # Dimension of each head
        self.causal_mask = causal_mask

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        # Weight matrices for Q, K, V for all heads combined
        self.W_q = nn.Parameter(torch.randn(d_model, n_heads * self.d_head))
        self.W_k = nn.Parameter(torch.randn(d_model, n_heads * self.d_head))
        self.W_v = nn.Parameter(torch.randn(d_model, n_heads * self.d_head))

        # Output projection matrix
        self.W_o = nn.Parameter(torch.randn(n_heads * self.d_head, d_model))

    def forward(self, inputs, mask=None):
        n_tokens = inputs.shape[0] # Sequence length

        # Compute Q, K, V for all heads simultaneously
        # (n_tokens, d_model) @ (d_model, n_heads * d_head) -> (n_tokens, n_heads * d_head)
        Q = inputs @ self.W_q
        K = inputs @ self.W_k
        V = inputs @ self.W_v

        # Reshape Q, K, V to separate the heads and prepare for batch matrix multiplication
        # From (n_tokens, n_heads * d_head) to (n_tokens, n_heads, d_head)
        Q = Q.reshape(n_tokens, self.n_heads, self.d_head)
        K = K.reshape(n_tokens, self.n_heads, self.d_head)
        V = V.reshape(n_tokens, self.n_heads, self.d_head)

        # Permute Q, K, V to have heads as the batch dimension for efficient processing
        # From (n_tokens, n_heads, d_head) to (n_heads, n_tokens, d_head)
        Q = Q.permute(1, 0, 2)
        K = K.permute(1, 0, 2)
        V = V.permute(1, 0, 2)

        # Transpose K for matrix multiplication (QK^T)
        KT = K.transpose(-1, -2)

        # Compute attention scores for all heads simultaneously
        # Result is (n_heads, n_tokens, n_tokens)
        scores = Q @ KT

        # Scale scores by the square root of d_head
        d_head_sqrt = torch.sqrt(torch.tensor(self.d_head, dtype=torch.float32))
        scores = scores / d_head_sqrt

        # Apply causal masking if enabled and a mask is provided
        if self.causal_mask and mask is not None:
            # Mask is broadcasted across heads
            scores = scores.masked_fill(mask, float('-inf'))

        # Apply softmax to get attention weights
        # Result is (n_heads, n_tokens, n_tokens)
        weights = torch.softmax(scores, dim=-1)

        # Compute the context vector for all heads
        # Result is (n_heads, n_tokens, d_head)
        context_vector = weights @ V

        # Permute and reshape the context vector back to (n_tokens, d_model)
        # First, permute from (n_heads, n_tokens, d_head) to (n_tokens, n_heads, d_head)
        context_vector = context_vector.permute(1, 0, 2)
        # Then, reshape to concatenate the head outputs: (n_tokens, n_heads * d_head) which is (n_tokens, d_model)
        context_vector = context_vector.reshape(n_tokens, self.n_heads * self.d_head)

        # Project the combined context vector back to d_model dimension using W_o
        # Result is (n_tokens, d_model)
        output = context_vector @ self.W_o

        return output

### **Instantiating and Testing the MultiHeadAttention Class**

Now, let's create an instance of our `MultiHeadAttention` class and test it with the same `inputs` and `mask` we used previously. We'll also apply the residual connection, mirroring the structure of a Transformer block.

In [78]:
torch.manual_seed(42)

# We will use the 'inputs' tensor that was already defined and seeded earlier
# inputs_class_test = torch.randn(n_tokens, d_model) # Removed this line

# Create an instance of the MultiHeadAttention layer
mha_layer = MultiHeadAttention(d_model=d_model, n_heads=n_heads, causal_mask=True)

# Assign the weights from the step-by-step implementation to the MHA layer
# This ensures both implementations use identical weights for comparison
mha_layer.W_q = W_q
mha_layer.W_k = W_k
mha_layer.W_v = W_v
mha_layer.W_o = W_o

# Forward pass through the MHA layer using the same inputs as the 'no class' example
attention_output_class = mha_layer(inputs, mask=mask)

# Add residual connection
residual_class = inputs + attention_output_class

print("Input to MHA class (first 3 tokens):\n", inputs[:3])
print("\nOutput from MHA class (first 3 tokens):\n", attention_output_class[:3])
print("\nResidual connection output (first 3 tokens):\n", residual_class[:3])

print(f"\nInput shape: {inputs.shape}")
print(f"Output shape from MHA class: {attention_output_class.shape}")
print(f"Residual output shape: {residual_class.shape}")

Input to MHA class (first 3 tokens):
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,  1.6806]])

Output from MHA class (first 3 tokens):
 tensor([[-12.5179,   0.8559,  -3.0215,  -4.5113,   0.0468,  -1.1004,  11.7385,
         -10.8862],
        [-13.6099,  -3.9001, -13.2319,  -7.3118,  -0.2075,  -3.0519,  22.3829,
          -9.2268],
        [  1.6282,   3.4164,   8.0239,   8.3214,  -1.4803,   7.0992, -15.6295,
           1.4433]], grad_fn=<SliceBackward0>)

Residual connection output (first 3 tokens):
 tensor([[-10.5909,   2.3432,  -2.1207,  -6.6168,   0.7252,  -2.3350,  11.6954,
         -12.4908],
        [-14.3620,  -2.2513, -13.6243,  -8.7154,  -0.9354,  -3.6114,  21.6141,
          -8.4644],
        [  3.2705,   3.2568,   7.5265,   8.7610,  -2.2385,   8.1775, -14.8287,
           3.1239]], grad

### **Comparison and Advantages of Class-based Implementation**

Comparing the step-by-step implementation with the class-based approach reveals several advantages:

1.  **Modularity**: The `MultiHeadAttention` class encapsulates all the logic and parameters, making the code cleaner and easier to manage.
2.  **Reusability**: Once defined, the class can be instantiated multiple times with different parameters, making it simple to create Transformer blocks with varying configurations or stack multiple attention layers.
3.  **Readability**: The `forward` method clearly outlines the flow of data through the attention mechanism.
4.  **Integration with PyTorch Ecosystem**: Being a `nn.Module`, it seamlessly integrates with PyTorch's training loop, optimizers, and other components, allowing for automatic gradient computation and easy model construction.
5.  **Maintainability**: Changes or improvements to the attention mechanism only need to be made in one place (the class definition).

In [79]:
residual_class

tensor([[-10.5909,   2.3432,  -2.1207,  -6.6168,   0.7252,  -2.3350,  11.6954,
         -12.4908],
        [-14.3620,  -2.2513, -13.6243,  -8.7154,  -0.9354,  -3.6114,  21.6141,
          -8.4644],
        [  3.2705,   3.2568,   7.5265,   8.7610,  -2.2385,   8.1775, -14.8287,
           3.1239],
        [  3.3433,   4.6284,   8.3512,   9.5822,  -1.7428,   7.1500, -15.9255,
           2.3841],
        [ -7.2523,   7.5504,  16.6206,   6.1644,   3.7888,   1.8357,  -9.4678,
          -1.9809],
        [-10.4061,   8.3981,  18.3978,   7.1894,   1.0644,   9.9126, -17.4105,
          -1.5298]], grad_fn=<AddBackward0>)

In [80]:
residual

tensor([[-10.5909,   2.3432,  -2.1207,  -6.6168,   0.7252,  -2.3350,  11.6954,
         -12.4908],
        [-14.3620,  -2.2513, -13.6243,  -8.7154,  -0.9354,  -3.6114,  21.6141,
          -8.4644],
        [  3.2705,   3.2568,   7.5265,   8.7610,  -2.2385,   8.1775, -14.8287,
           3.1239],
        [  3.3433,   4.6284,   8.3512,   9.5822,  -1.7428,   7.1500, -15.9255,
           2.3841],
        [ -7.2523,   7.5504,  16.6206,   6.1644,   3.7888,   1.8357,  -9.4678,
          -1.9809],
        [-10.4061,   8.3981,  18.3978,   7.1894,   1.0644,   9.9126, -17.4105,
          -1.5298]], grad_fn=<AddBackward0>)